# Tumor Preprocessing

- **Purpose:** Interactive preprocessing of GSE178341: attach donor, chemistry, and epithelial-state metadata, define the `donor_chemistry` batch key, and subset to epithelial tumor cells and the shared genes.
- **Inputs:** GSE178341 H5, metadata TSV, cell-state label TSV, `data/processed/genes_combined.csv`.
- **Outputs:** `data/processed/preprocessed_tumor.h5ad` (written by `scripts/03_tumor_pp.py`).
- **Result:** 108,131 epithelial tumor cells with the metadata used by the integration.


## Overview
This notebook preprocesses the tumor dataset SCP1162 in preperation for CRC cancer analysis. This notebook does the following: 

1. Load the data
2. Checking out the h5py file and creating the necessary anndata object
3. Add the cell metadata
4. Add the cell state labels and the batch_key batch to the adata
    - In the tumor dataset the batch_key corresponds to a the donor_chemistry. 
    - Examples:
        - C162_v2 and C162_v3
        - Donors with two sample like this: C171_TA_v3 and C171_TB_v3 are keept as only one batch: C171_v3
6. Add some more donor metadata from the Paper
7. Subset to epiT cells with unique gene IDs
    - Also checking out some interesting details for duplicated non-versioned gene IDs
8. Subset to the gene intersection with the cell lines
    - One donor sequenced with both 10X 3' chemistries: v2 and v3

In [1]:
import pandas as pd
import anndata as ad
import numpy as np
import os

from scipy import sparse

import h5py

In [2]:
DATA_DIR = os.path.join("..", "..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

h5_file_path = os.path.join(DATA_DIR, "raw", "GSE178341_crc10x_full_c295v4_submit.h5")
metadata_path = os.path.join(DATA_DIR, "raw", "metatable_v3_fix_v3.tsv")
label_path = os.path.join(DATA_DIR, "raw", "crc10x_tSNE_cl_global.tsv") 
gene_file_path = os.path.join(PROCESSED_DIR, "genes_combined.csv")

# Exploration of the h5 data

In [3]:
data = h5py.File(h5_file_path, "r")

In [4]:
data["matrix"].keys()

<KeysViewHDF5 ['barcodes', 'data', 'features', 'indices', 'indptr', 'shape']>

In [5]:
data["matrix"]["features"].keys()

<KeysViewHDF5 ['_all_tag_keys', 'feature_type', 'genome', 'id', 'name']>

In [6]:
matrix_group = data["matrix"]
counts = matrix_group["data"][:]
indices = matrix_group["indices"][:]
indptr = matrix_group["indptr"][:]
shape = tuple(matrix_group["shape"][:])

features = matrix_group["features"]
gene_ids = [x.decode() if isinstance(x, bytes) else x for x in features["id"][:]]
gene_names = [x.decode() if isinstance(x, bytes) else x for x in features["name"][:]]
genome = [x.decode() if isinstance(x, bytes) else x for x in features["genome"][:]] # 10x reference package name; hg19 from 2009 but liftover version28
feature_types = [x.decode() if isinstance(x, bytes) else x for x in features["feature_type"][:]]
cell_barcodes = [x.decode() if isinstance(x, bytes) else x for x in matrix_group["barcodes"][:]]

print(f"Shape: {shape[0]} genes x {shape[1]} cells")
print(f"10x reference package name: {pd.Series(genome).unique()}")
print(f"Features consisting only of: {pd.Series(feature_types).unique()}")
print(f"Number of unique gene names: {len(pd.unique(pd.Series(gene_names)))}")
print(f"Number of unique gene ids: {len(pd.unique(pd.Series(gene_ids)))}")

Shape: 43113 genes x 370115 cells
10x reference package name: ['GRCh37_liftover_v28']
Features consisting only of: ['Gene Expression']
Number of unique gene names: 41195
Number of unique gene ids: 43113


In [7]:
mat = sparse.csc_matrix((counts, indices, indptr), shape=shape, dtype="uint32")
obs = pd.DataFrame(index=cell_barcodes)
var_id = pd.DataFrame(index=gene_ids) # gene_symbol not unique -> use id with geneAnno later

adata = ad.AnnData(X=mat.T, obs=obs, var=var_id)

In [8]:
print("Shape of adata: ", adata.shape)
print("Shape of h5 cells x genes: ", (len(cell_barcodes), len(gene_ids)))

Shape of adata:  (370115, 43113)
Shape of h5 cells x genes:  (370115, 43113)


In [9]:
adata.var["gene_symbol"] = gene_names
adata.var["gene_id"] = adata.var.index
adata.obs["cell_id"] = adata.obs.index
adata.obs["donor_id"] = adata.obs["cell_id"].str.extract(r'^(C\d{2,4})')
adata.var["gene_id"] = adata.var["gene_id"].str.split(".").str[0]

In [10]:
print(adata.obs["donor_id"].notna().all())
adata.obs["donor_id"].value_counts()

True


donor_id
C162    22587
C124    16945
C171    16199
C123    14572
C126    12914
        ...  
C169     2245
C115     1995
C172     1542
C119     1534
C167     1443
Name: count, Length: 62, dtype: int64

# Adding Metadata

In [11]:
metadata = pd.read_csv(
        metadata_path,
        sep="\t",
        usecols=["NAME", "donor_id", "MMRStatusTumor", "library_preparation_protocol__ontology_label", "biosample_id"]
).iloc[1:]
metadata = metadata.rename(columns = {"NAME":"cell_id", "library_preparation_protocol__ontology_label": "chemistry", "MMRStatusTumor":"mmr_status"})

In [12]:
metadata.head(3)

,cell_id,biosample_id,donor_id,mmr_status,chemistry
1,C103_T_1_1_0_c1_v2_id-AAACCTGCATGCTAGT,C103_T_1_1_0_c1_v2,C103,MMRp,10X 3' v2 sequencing
2,C103_T_1_1_0_c1_v2_id-AAACCTGGTAGCCTAT,C103_T_1_1_0_c1_v2,C103,MMRp,10X 3' v2 sequencing
3,C103_T_1_1_0_c1_v2_id-AAACCTGGTTGTCGCG,C103_T_1_1_0_c1_v2,C103,MMRp,10X 3' v2 sequencing


In [13]:
metadata["chemistry"].value_counts()

chemistry
10X 3' v2 sequencing    267712
10x 3' v3 sequencing    103511
Name: count, dtype: int64

In [14]:
metadata["chemistry"] = np.where(
    metadata["chemistry"].str.contains("v2", na=False),
    "v2",
    np.where(
        metadata["chemistry"].str.contains("v3", na=False),
        "v3",
        metadata["chemistry"]
    )
)

In [15]:
metadata["chemistry"].value_counts()

chemistry
v2    267712
v3    103511
Name: count, dtype: int64

In [16]:
new_obs = adata.obs.merge(metadata, on=["cell_id", "donor_id"], how="inner")
new_obs.index = new_obs["cell_id"]
new_obs.index.name = None
adata.obs = new_obs

# Adding Cell state labels and batch_key batch

In [17]:
labels = pd.read_csv(label_path, sep="\t", usecols=["NAME", "ClusterFull", "ClusterMidway"]).iloc[1:]
labels = labels.rename(columns = {"NAME":"cell_id", "ClusterFull":"label", "ClusterMidway":"cell_type"})
extracted = adata.obs['biosample_id'].str.extract(r'^(?P<donor>C\d+)_(?P<sample>[A-Z]+)_.*_(?P<chemistry>v\d+)$')

In [18]:
labels.head(3)

,cell_id,label,cell_type
1,C103_T_1_1_0_c1_v2_id-AAACCTGCATGCTAGT,Tumor cE01 (Stem/TA-like),EpiT
2,C103_T_1_1_0_c1_v2_id-AAACCTGGTAGCCTAT,Tumor cE01 (Stem/TA-like),EpiT
3,C103_T_1_1_0_c1_v2_id-AAACCTGGTTGTCGCG,Tumor cE03 (Stem/TA-like prolif),EpiT


In [19]:
extracted.head(3)

,donor,sample,chemistry
C103_T_1_1_0_c1_v2_id-AAACCTGCATGCTAGT,C103,T,v2
C103_T_1_1_0_c1_v2_id-AAACCTGGTAGCCTAT,C103,T,v2
C103_T_1_1_0_c1_v2_id-AAACCTGGTTGTCGCG,C103,T,v2


In [20]:
print(f"Number of missing donors: {extracted['donor'].isna().sum()}")
print(f"Number of missing samples: {extracted['sample'].isna().sum()}")
print(f"Number of missing chemistries: {extracted['chemistry'].isna().sum()}")

Number of missing donors: 0
Number of missing samples: 0
Number of missing chemistries: 0


In [21]:
batch_style = "donor_chemistry"
if batch_style == "donor_chemistry":
    # C130/C171 with two lesions (TA/TB) are treated as the same donor biology,
    # matching the batch design used by the HPO.
    def make_batch(row):
        return f"{row['donor']}_{row['chemistry']}"
elif batch_style == "donor_sample_chemistry":
    # keep lesions of C130/C171 as separate batches: As the first idea but should be discard 
    # according to Luecken scBestPracties-Book: Data Integration (Which batch label box)
    def make_batch(row):
        if row["sample"] == "T":
            return f"{row['donor']}_{row['chemistry']}"
        else:
            return f"{row['donor']}_{row['sample']}_{row['chemistry']}"
else:
    raise ValueError(f"Unknown batch_style: {batch_style}. Use 'donor_chemistry' or 'donor_sample_chemistry'.")

adata.obs["batch"] = extracted.apply(make_batch, axis=1)

In [22]:
adata.obs["batch"].value_counts()

batch
C124_v2    16945
C171_v3    16199
C123_v2    14572
C126_v2    12914
C125_v2    12660
           ...  
C169_v3     2245
C115_v2     1995
C172_v3     1542
C119_v2     1534
C167_v3     1443
Name: count, Length: 63, dtype: int64

In [23]:
new_obs = adata.obs.merge(labels, on="cell_id", how="inner")
new_obs.index = new_obs["cell_id"]
new_obs.index.name = None
adata.obs = new_obs

In [24]:
print(f"Number of NA batches: {adata.obs['batch'].isna().sum()}")
extracted["sample"].value_counts()

Number of NA batches: 0


sample
T     234231
N     112864
TA     13305
TB      9715
Name: count, dtype: int64

In [25]:
adata

AnnData object with n_obs × n_vars = 370115 × 43113
    obs: 'cell_id', 'donor_id', 'biosample_id', 'mmr_status', 'chemistry', 'batch', 'label', 'cell_type'
    var: 'gene_symbol', 'gene_id'

# Adding Paper Metadata

In [26]:
donor_metadata = {
    "C106":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "male", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "low"},
    "C109":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C110":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "high"},
    "C111":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C114":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C115":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C116":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C118":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "male", "tissue_site": "right", "pathological_stage": 2, "tumor_grade": "low"},
    "C119":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "high"},
    "C122":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C123":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C130":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "low"},
    "C132":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "female", "tissue_site": "left", "pathological_stage": 4, "tumor_grade": "high"},
    "C137":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "high"},
    "C138":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C139":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "low"},
    "C142":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C143":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C144":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C146":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C147":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "low"},
    "C151":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "high"},
    "C152":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 2, "tumor_grade": "low"},
    "C154":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 2, "tumor_grade": "low"},
    "C156":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C158":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C163":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "high"},
    "C164":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "low"},
    "C165":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "high"},
    "C167":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "high"},
    "C168":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "male", "tissue_site": "right", "pathological_stage": 2, "tumor_grade": "low"},
    "C169":  {"MMR_status": "MMRd", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 2, "tumor_grade": "low"},
    "C170":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C173":  {"MMR_status": "MMRd", "mlh1_status": True, "sex": "female", "tissue_site": "left", "pathological_stage": 4, "tumor_grade": "high"},

    "C103":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "left", "pathological_stage": 2, "tumor_grade": "low"},
    "C104":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C105":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "low"},
    "C107":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C112":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C113":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 2, "tumor_grade": "low"},
    "C124":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "left", "pathological_stage": 4, "tumor_grade": "high"},
    "C125":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "left", "pathological_stage": 3, "tumor_grade": "low"},
    "C126":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "low"},
    "C129":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "left", "pathological_stage": 3, "tumor_grade": "low"},
    "C133":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "low"},
    "C134":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "left", "pathological_stage": 4, "tumor_grade": "low"},
    "C135":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 1, "tumor_grade": "low"},
    "C136":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "left", "pathological_stage": 3, "tumor_grade": "low"},
    "C140":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "left", "pathological_stage": 1, "tumor_grade": "low"},
    "C145":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "high"},
    "C149":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "low"},
    "C150":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "left", "pathological_stage": 3, "tumor_grade": "low"},
    "C153":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "left", "pathological_stage": 2, "tumor_grade": "low"},
    "C155":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C157":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "left", "pathological_stage": 4, "tumor_grade": "low"},
    "C159":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "high"},
    "C160":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "high"},
    "C161":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
    "C162":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 4, "tumor_grade": "low"},
    "C166":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "left", "pathological_stage": 3, "tumor_grade": "low"},
    "C171":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "male", "tissue_site": "right", "pathological_stage": 2, "tumor_grade": "low"},
    "C172":  {"MMR_status": "MMRp", "mlh1_status": False, "sex": "female", "tissue_site": "right", "pathological_stage": 3, "tumor_grade": "low"},
}

df_donor_meta = pd.DataFrame.from_dict(donor_metadata).T
df_donor_meta.head(3)

,MMR_status,mlh1_status,sex,tissue_site,pathological_stage,tumor_grade
C106,MMRd,True,male,right,4,low
C109,MMRd,True,female,right,3,low
C110,MMRd,True,male,right,3,high


In [27]:
for column in df_donor_meta.columns:
    if column == "MMR_status":
        continue
    adata.obs[column] = adata.obs["donor_id"].map(df_donor_meta[column])

adata.obs.head(3)

,cell_id,donor_id,biosample_id,mmr_status,chemistry,batch,label,cell_type,mlh1_status,sex,tissue_site,pathological_stage,tumor_grade
C103_T_1_1_0_c1_v2_id-AAACCTGCATGCTAGT,C103_T_1_1_0_c1_v2_id-AAACCTGCATGCTAGT,C103,C103_T_1_1_0_c1_v2,MMRp,v2,C103_v2,Tumor cE01 (Stem/TA-like),EpiT,False,male,left,2,low
C103_T_1_1_0_c1_v2_id-AAACCTGGTAGCCTAT,C103_T_1_1_0_c1_v2_id-AAACCTGGTAGCCTAT,C103,C103_T_1_1_0_c1_v2,MMRp,v2,C103_v2,Tumor cE01 (Stem/TA-like),EpiT,False,male,left,2,low
C103_T_1_1_0_c1_v2_id-AAACCTGGTTGTCGCG,C103_T_1_1_0_c1_v2_id-AAACCTGGTTGTCGCG,C103,C103_T_1_1_0_c1_v2,MMRp,v2,C103_v2,Tumor cE03 (Stem/TA-like prolif),EpiT,False,male,left,2,low


# Trim to only epiT cells and unique gene IDs

In [28]:
adata.obs["cell_type"].value_counts()

cell_type
EpiT            108131
Epi              60164
Plasma           37809
TCD4             34598
B                25660
TCD8             23486
Macro            20280
Mono             14242
Tgd               9383
Endo              7520
DC                5549
Fibro             5231
TZBTB16           4742
NK                3924
Mast              3834
Granulo           2043
Peri              1525
SmoothMuscle       881
ILC                832
Schwann            281
Name: count, dtype: int64

In [29]:
adata.obs["label"].value_counts()

label
Tumor cE01 (Stem/TA-like)           49252
Tumor cE03 (Stem/TA-like prolif)    37957
cP1 (Plasma IgA)                    20305
cM02 (Macrophage-like)              20280
cP2 (Plasma IgG)                    16438
                                    ...  
cS14 (Endo)                           146
cS07 (Endo capillary-like)            100
cM08 (AS-DC)                           94
cS20 (Pericyte prolif)                 44
Tumor cE07 (Goblet/Enterocyte)         36
Name: count, Length: 98, dtype: int64

In [30]:
adata.obs.cell_type.value_counts().to_frame().style

,count
cell_type,
EpiT,108131
Epi,60164
Plasma,37809
TCD4,34598
B,25660
TCD8,23486
Macro,20280
Mono,14242
Tgd,9383


In [31]:
mask_epiT = (adata.obs["cell_type"] == "EpiT").values
mask_epiT.sum()

np.int64(108131)

## Creating unique gene IDs without version numbers
We can see that all the PAR_Y (and therefore each second occurence of the non-versioned gene_id) have zero counts compared to the non-par_y genes. We therefore just remove the duplicates and keep only the first occurence for each gene_id.

In [32]:
adata.var[adata.var["gene_id"].duplicated(keep=False)].sort_values("gene_id")

,gene_symbol,gene_id
ENSG00000002586.19_4,CD99,ENSG00000002586
ENSG00000002586.19_4_PAR_Y,CD99,ENSG00000002586
ENSG00000124333.15_2,VAMP7,ENSG00000124333
ENSG00000124333.15_2_PAR_Y,VAMP7,ENSG00000124333
ENSG00000124334.17_4,IL9R,ENSG00000124334
ENSG00000124334.17_4_PAR_Y,IL9R,ENSG00000124334
ENSG00000167393.17_3_PAR_Y,PPP2R3B,ENSG00000167393
ENSG00000167393.17_3,PPP2R3B,ENSG00000167393
ENSG00000168939.11_2_PAR_Y,SPRY3,ENSG00000168939
ENSG00000168939.11_2,SPRY3,ENSG00000168939


In [33]:
dup_mask = adata.var["gene_id"].duplicated(keep=False).values
X_dup = adata.X[:, dup_mask]

dup_gene_ids = adata.var.loc[dup_mask, "gene_id"]
totals = X_dup.sum(axis=0).A1

dup_df = pd.DataFrame({
        "gene_id": dup_gene_ids,
        "total_counts": totals
    })

counts_dup = (dup_df.groupby(dup_df.index)["total_counts"].sum())
counts_dup

ENSG00000002586.19_4           587372
ENSG00000002586.19_4_PAR_Y          0
ENSG00000124333.15_2            80050
ENSG00000124333.15_2_PAR_Y          0
ENSG00000124334.17_4             1283
ENSG00000124334.17_4_PAR_Y          0
ENSG00000167393.17_3            21241
ENSG00000167393.17_3_PAR_Y          0
ENSG00000168939.11_2             3794
ENSG00000168939.11_2_PAR_Y          0
ENSG00000169084.13_3            93218
ENSG00000169084.13_3_PAR_Y          0
ENSG00000169093.15_3            52895
ENSG00000169093.15_3_PAR_Y          0
ENSG00000169100.13_2          3086081
ENSG00000169100.13_2_PAR_Y          0
ENSG00000178605.13_2           136242
ENSG00000178605.13_2_PAR_Y          0
ENSG00000182162.10_3            10750
ENSG00000182162.10_3_PAR_Y          0
ENSG00000182378.13_5            29904
ENSG00000182378.13_5_PAR_Y          0
ENSG00000185203.12_4              286
ENSG00000185203.12_4_PAR_Y          0
ENSG00000185291.11_2            32340
ENSG00000185291.11_2_PAR_Y          0
ENSG00000185

In [34]:
mask_duplicated_genes = adata.var["gene_id"].duplicated().values
mask_duplicated_genes.sum()

np.int64(28)

In [35]:
X_filtered = adata.X[mask_epiT][:, ~mask_duplicated_genes].copy()
obs_filtered = adata.obs.loc[mask_epiT].copy()
var_filtered = adata.var.loc[~mask_duplicated_genes].copy()

In [36]:
adata = ad.AnnData(X=X_filtered, obs=obs_filtered, var=var_filtered)
adata.var.index = adata.var["gene_id"]
adata.var.index.name = None

In [37]:
adata.var.head(3)

,gene_symbol,gene_id
ENSG00000243485,RP11-34P13.3,ENSG00000243485
ENSG00000237613,FAM138A,ENSG00000237613
ENSG00000186092,OR4F5,ENSG00000186092


In [38]:
print(f"Gene_id is unique?: {adata.var['gene_id'].is_unique}")
print(f"Gene_symbol is unique?: {adata.var['gene_symbol'].is_unique}")

Gene_id is unique?: True
Gene_symbol is unique?: False


# Subset to gene intersection with the cell line genes

In [39]:
gene_df = pd.read_csv(gene_file_path)
gene_df.head(3)

,gene_id,gene_symbol
0,ENSG00000121410,A1BG
1,ENSG00000268895,A1BG-AS1
2,ENSG00000148584,A1CF


In [40]:
print("Are all genes from the intersection present: ", \
      gene_df["gene_id"].isin(adata.var["gene_id"]).all())

Are all genes from the intersection present:  True


In [41]:
gene_mask = adata.var["gene_id"].isin(gene_df["gene_id"])
adata = adata[:, gene_mask].copy()
adata.var = adata.var.set_index("gene_symbol", drop=False)
adata.var.index.name = None

In [42]:
adata.var.head(3)

,gene_symbol,gene_id
FAM138A,FAM138A,ENSG00000237613
OR4F5,OR4F5,ENSG00000186092
RP11-34P13.7,RP11-34P13.7,ENSG00000238009


In [43]:
adata.obs.batch.value_counts().to_frame().sort_index().style

,count
batch,
C103_v2,4580
C104_v2,1694
C105_v2,1388
C106_v2,1113
C107_v2,2156
C109_v2,1511
C110_v2,790
C111_v2,650
C112_v2,1760


## One donor Sequenced with both chemistries

In [44]:
chemistry_counts = adata.obs.groupby("donor_id")["chemistry"].nunique().loc[lambda x: x > 1]
chemistry_counts

donor_id
C162    2
Name: chemistry, dtype: int64

In [45]:
adata.obs["c_value"] = adata.obs["biosample_id"].str.extract(r"_(c\d+)_")

adata.obs["c_value"].value_counts()

c_value
c1    100485
c2      7646
Name: count, dtype: int64

In [46]:
adata.obs.loc[adata.obs["c_value"] == "c2", "donor_id"].unique()

array(['C111', 'C125', 'C126', 'C133', 'C134', 'C135', 'C136', 'C137',
       'C138', 'C139', 'C140', 'C162'], dtype=object)

In [47]:
adata.obs[adata.obs["c_value"] == "c2"].groupby("donor_id").size()

donor_id
C111     110
C125     460
C126     163
C133     489
C134     631
C135     178
C136     287
C137     116
C138      30
C139    1256
C140     381
C162    3545
dtype: int64

In [48]:
adata.obs[adata.obs["donor_id"].isin(chemistry_counts.index)].sort_values(["donor_id", "chemistry"]).head(3)

,cell_id,donor_id,biosample_id,mmr_status,chemistry,batch,label,cell_type,mlh1_status,sex,tissue_site,pathological_stage,tumor_grade,c_value
C162_T_0_0_0_c2_v2_id-AAACCTGAGAATTCCC,C162_T_0_0_0_c2_v2_id-AAACCTGAGAATTCCC,C162,C162_T_0_0_0_c2_v2,MMRp,v2,C162_v2,Tumor cE01 (Stem/TA-like),EpiT,False,male,right,4,low,c2
C162_T_0_0_0_c2_v2_id-AAACCTGAGACAATAC,C162_T_0_0_0_c2_v2_id-AAACCTGAGACAATAC,C162,C162_T_0_0_0_c2_v2,MMRp,v2,C162_v2,Tumor cE01 (Stem/TA-like),EpiT,False,male,right,4,low,c2
C162_T_0_0_0_c2_v2_id-AAACCTGCATCCTTGC,C162_T_0_0_0_c2_v2_id-AAACCTGCATCCTTGC,C162,C162_T_0_0_0_c2_v2,MMRp,v2,C162_v2,Tumor cE01 (Stem/TA-like),EpiT,False,male,right,4,low,c2


In [49]:
adata.obs[adata.obs["donor_id"].isin(chemistry_counts.index)].sort_values(["donor_id", "chemistry"]).tail(3)

,cell_id,donor_id,biosample_id,mmr_status,chemistry,batch,label,cell_type,mlh1_status,sex,tissue_site,pathological_stage,tumor_grade,c_value
C162_T_0_0_1_c1_v3_id-TTTGTTGAGCACACAG,C162_T_0_0_1_c1_v3_id-TTTGTTGAGCACACAG,C162,C162_T_0_0_1_c1_v3,MMRp,v3,C162_v3,Tumor cE02 (Stem/TA-like/Immature Goblet),EpiT,False,male,right,4,low,c1
C162_T_0_0_1_c1_v3_id-TTTGTTGAGTACAGCG,C162_T_0_0_1_c1_v3_id-TTTGTTGAGTACAGCG,C162,C162_T_0_0_1_c1_v3,MMRp,v3,C162_v3,Tumor cE01 (Stem/TA-like),EpiT,False,male,right,4,low,c1
C162_T_0_0_1_c1_v3_id-TTTGTTGGTCCGTACG,C162_T_0_0_1_c1_v3_id-TTTGTTGGTCCGTACG,C162,C162_T_0_0_1_c1_v3,MMRp,v3,C162_v3,Tumor cE01 (Stem/TA-like),EpiT,False,male,right,4,low,c1


In [50]:
print(f"is view?: {adata.is_view}")
print(f"var index unique?: {adata.var.index.is_unique}")
print(f"obs index unique?: {adata.obs.index.is_unique}\n")
print(adata.obs.dtypes)
print("\n", adata.var.dtypes)
print(f"\n X type: {adata.X.dtype}")

is view?: False
var index unique?: True
obs index unique?: True

cell_id               object
donor_id              object
biosample_id          object
mmr_status            object
chemistry             object
batch                 object
label                 object
cell_type             object
mlh1_status           object
sex                   object
tissue_site           object
pathological_stage    object
tumor_grade           object
c_value               object
dtype: object

 gene_symbol    object
gene_id        object
dtype: object

 X type: uint32


In [51]:
adata.obs["dataset"] = "tumor"
adata.obs["cell_line"] = "tumor"

In [52]:
cols = adata.obs.select_dtypes(include=["object"]).columns.tolist()
cols.remove("cell_id")
for col in cols:
    adata.obs[col] = adata.obs[col].astype("category")

adata.obs["mlh1_status"] = adata.obs["mlh1_status"].astype(bool)

In [53]:
print(f"is view?: {adata.is_view}")
print(f"var index unique?: {adata.var.index.is_unique}")
print(f"obs index unique?: {adata.obs.index.is_unique}\n")
print(adata.obs.dtypes)
print("\n", adata.var.dtypes)
print(f"\n X type: {adata.X.dtype}")

is view?: False
var index unique?: True
obs index unique?: True

cell_id                 object
donor_id              category
biosample_id          category
mmr_status            category
chemistry             category
batch                 category
label                 category
cell_type             category
mlh1_status               bool
sex                   category
tissue_site           category
pathological_stage    category
tumor_grade           category
c_value               category
dataset               category
cell_line             category
dtype: object

 gene_symbol    object
gene_id        object
dtype: object

 X type: uint32


In [54]:
adata.obs.head(3)

,cell_id,donor_id,biosample_id,mmr_status,chemistry,batch,label,cell_type,mlh1_status,sex,tissue_site,pathological_stage,tumor_grade,c_value,dataset,cell_line
C103_T_1_1_0_c1_v2_id-AAACCTGCATGCTAGT,C103_T_1_1_0_c1_v2_id-AAACCTGCATGCTAGT,C103,C103_T_1_1_0_c1_v2,MMRp,v2,C103_v2,Tumor cE01 (Stem/TA-like),EpiT,False,male,left,2,low,c1,tumor,tumor
C103_T_1_1_0_c1_v2_id-AAACCTGGTAGCCTAT,C103_T_1_1_0_c1_v2_id-AAACCTGGTAGCCTAT,C103,C103_T_1_1_0_c1_v2,MMRp,v2,C103_v2,Tumor cE01 (Stem/TA-like),EpiT,False,male,left,2,low,c1,tumor,tumor
C103_T_1_1_0_c1_v2_id-AAACCTGGTTGTCGCG,C103_T_1_1_0_c1_v2_id-AAACCTGGTTGTCGCG,C103,C103_T_1_1_0_c1_v2,MMRp,v2,C103_v2,Tumor cE03 (Stem/TA-like prolif),EpiT,False,male,left,2,low,c1,tumor,tumor
